# 03 — Preprocessing & Training Model Baseline: Server VPS PT Hotdog Water Sport

**Data:** rekaman lalu lintas server VPS PT Hotdog Water Sport (lalu lintas sebenarnya: pengunjung situs web perusahaan dan serangan dari internet). Tidak ada simulasi serangan di server ini.

**Isi notebook (urut):**
1. **Preprocessing**: rekaman `.pcap.gz` (header paket per jam) diputar ulang dengan **pipa yang sama persis dengan
   sistem NIDS** (penyusun aliran + fitur agregasi berjalan) → tabel 68 fitur aliran + 7 fitur agregasi.
2. Gambaran lalu lintas server.
3. Pelabelan sederhana berbasis aturan (untuk membersihkan data latih dan memeriksa hasil).
4. Training Model A + C pada periode latih: **apa adanya** vs **dibersihkan**.
5. Pemeriksaan pada periode uji.
6. Training **model final** dari seluruh data (dibersihkan) → berkas model yang dipasang di sistem NIDS.

Rancangan model mengikuti hasil penalaan (notebook `07_Penalaan_Model_VPS_Perusahaan.ipynb`): fitur diberi
**transformasi log** sebelum distandarkan, data latih dibersihkan dari **sumber serangan intens dan pemindaian
terdistribusi per subnet**, *contamination* A = 0,001 dan C = 0,0075, jendela 300 detik, n_estimators = 100,
max_samples = 256. Rancangan dua model A+C divalidasi pada CIC-IDS2017 (notebook 02 dan 07); model yang dipasang di
server dilatih dari lalu lintas server itu sendiri (Cantone et al., 2024).

In [ ]:
import os, sys, glob, json, time
import numpy as np
import pandas as pd
import joblib

FOLDER = os.path.abspath('')                                     # folder notebook ini (04_Kode/4_notebook)
KODE = os.path.abspath(os.path.join(FOLDER, '..', '2_sistem_nids'))   # kode sistem NIDS
DATA = os.path.abspath(os.path.join(FOLDER, '..', '..', '05_Data_Eksperimen', 'Server-HotDog'))   # rekaman & hasil
sys.path.insert(0, KODE)
from nids.pipa import putar_ulang, KOLOM                         # pipa yang SAMA dengan deteksi real-time
from nids.model import latih, ModelDeteksi
from nids.agregasi import FITUR_AGREGASI, subnet
from nids.aliran import FITUR as FITUR_ALIRAN
import latih_model as LM                                         # aturan pembersihan yang sama dengan sistem NIDS
pd.set_option('display.max_columns', 30)

# ---- konfigurasi ----
NAMA = 'vps_perusahaan'
NAMA_SERVER = 'Server VPS PT Hotdog Water Sport'
REKAMAN = os.path.join(DATA, 'rekaman')
HASIL = os.path.join(DATA, 'hasil')
os.makedirs(HASIL, exist_ok=True)
SERVER_IP = {'2a02:4780:5e:4c51::1', '192.0.2.20'}                # alamat IPv4 & IPv6 server (lalu lintas dari alamat lain = "masuk")
JENDELA = 300                            # detik, jendela fitur agregasi
C_A, C_C = 0.001, 0.0075                 # contamination Model A dan Model C (hasil penalaan notebook 07)
TANGGAL_UJI = '2026-10-27'             # UTC: sebelum waktu ini = latih, sejak waktu ini = uji
DARI = None                          # UTC: hanya rekaman sejak waktu ini yang dipakai (None = semua)
print('Kode NIDS :', KODE)
print('Data      :', DATA)
print('Rekaman   :', len(glob.glob(os.path.join(REKAMAN, '*.pcap.gz'))), 'berkas')

## 1. Preprocessing: rekaman → aliran berfitur

Setiap berkas berisi satu jam header paket (payload sudah dikosongkan, panjang aslinya tetap tercatat).
Paket dikelompokkan menjadi **aliran dua arah** dengan aturan CICFlowMeter (timeout 120 detik, ditutup oleh FIN/RST),
lalu setiap aliran yang selesai diberi 68 fitur aliran dan 7 fitur agregasi **300 detik terakhir** per IP sumber dan
per subnet. Karena memakai kode yang sama dengan deteksi real-time, fitur di sini identik dengan yang dilihat model
saat sistem berjalan di server.

In [ ]:
berkas = sorted(glob.glob(os.path.join(REKAMAN, '*.pcap.gz')))
if DARI:                                 # nama berkas = waktu mulai rekam (UTC), mis. cap_20261105_072933
    batas_nama = 'cap_' + pd.Timestamp(DARI).strftime('%Y%m%d_%H%M%S')
    berkas = [b for b in berkas if os.path.basename(b) >= batas_nama]
t0 = time.time()
aliran, pipa = putar_ulang(berkas, JENDELA)
df = pd.DataFrame(aliran)
df = df[[c for c in KOLOM + ['_mulai_us'] if c in df.columns]]
FITUR_PARQUET = os.path.join(HASIL, 'fitur_%s.parquet' % NAMA)
df.to_parquet(FITUR_PARQUET, index=False)                       # bahan pasang-deteksi.sh --latih
df['Waktu'] = pd.to_datetime(df['Waktu Selesai'], unit='us', utc=True)
df['Masuk'] = ~df['Source IP'].isin(SERVER_IP)                   # koneksi dimulai dari luar
print('%d berkas | %s paket | %s aliran | %.0f detik' % (len(berkas), f'{pipa.jumlah_paket:,}', f'{len(df):,}', time.time() - t0))
print('Rentang (UTC): %s s.d. %s' % (df.Waktu.min().strftime('%d-%m-%Y %H:%M'), df.Waktu.max().strftime('%d-%m-%Y %H:%M')))
print('Tabel fitur disimpan:', FITUR_PARQUET)

In [ ]:
print('Contoh hasil preprocessing (sebagian kolom):')
df[['Waktu', 'Source IP', 'Destination Port', 'Protocol', 'Flow Duration', 'Total Fwd Packets',
    'Total Backward Packets', 'Total Length of Bwd Packets'] + list(FITUR_AGREGASI)].head(10)

## 2. Gambaran lalu lintas server

In [ ]:
m = df[df.Masuk]
ringkas = pd.DataFrame({
    'aliran masuk': m.groupby(m.Waktu.dt.date).size(),
    'IP sumber berbeda': m.groupby(m.Waktu.dt.date)['Source IP'].nunique(),
    'aliran keluar (dari server)': df[~df.Masuk].groupby(df[~df.Masuk].Waktu.dt.date).size(),
})
gagal = (m['Total Backward Packets'] == 0) | ((m['RST Flag Count'] > 0) & (m['Total Length of Bwd Packets'] == 0))
print('Aliran masuk yang ditolak / tidak dibalas: %.1f%%  (ciri khas pemindaian bot)' % (gagal.mean() * 100))
ringkas.index.name = 'tanggal (UTC)'
ringkas

In [ ]:
print('Port tujuan terbanyak (aliran masuk):')
m['Destination Port'].value_counts().head(10).to_frame('aliran')

## 3. Pelabelan sederhana (aturan)

Isolation Forest **tidak membutuhkan label** untuk dilatih. Label sederhana ini hanya dipakai untuk **membersihkan data
latih** dan **memeriksa hasil**:

| Kategori | Aturan |
|---|---|
| Serangan intens | pernah ≥ 20 port berbeda **atau** ≥ 20 koneksi SSH dalam 5 menit |
| Pemindaian terdistribusi | subnet IP itu (/24 IPv4, /64 IPv6) pernah ≥ 20 port berbeda **atau** ≥ 20 IP berbeda dalam 5 menit |
| Pemindai lambat | ≥ 10 port berbeda selama periode, tanpa pernah menerima data balasan |
| Pengunjung berdata | pernah bertukar data dengan server dan bukan kategori di atas (perkiraan lalu lintas **normal**) |
| Lainnya | sisanya, kebanyakan bot yang hanya mengetuk 1–2 port (kebisingan latar internet) |
| Server (keluar) | koneksi yang dimulai oleh server sendiri (mis. sinkronisasi waktu NTP, DNS) |

In [ ]:
def sumber_intens(g):
    mm = g[g.Masuk]
    return set(mm.loc[(mm.agg_ip_port_unik >= LM.AMBANG_SCAN) | (mm.agg_ip_aliran_ssh >= LM.AMBANG_SSH), 'Source IP'])

def kategori(g):
    """Kategori dihitung dari periode g saja (latih dan uji terpisah, tanpa bocoran)."""
    intens = sumber_intens(g)
    mm = g[g.Masuk].assign(Subnet=lambda x: x['Source IP'].map(subnet))
    sub_scan = set(mm.loc[(mm.agg_s24_port_unik >= LM.AMBANG_SCAN) | (mm.agg_s24_ip_unik >= LM.AMBANG_SCAN), 'Subnet'])
    per_ip = mm.groupby('Source IP').agg(port=('Destination Port', 'nunique'), sub=('Subnet', 'first'),
                                         data=('Total Length of Bwd Packets', lambda s: (s > 0).any()))
    k = pd.Series('Lainnya', index=per_ip.index)
    k[per_ip.data] = 'Pengunjung berdata'
    k[(per_ip.port >= 10) & ~per_ip.data] = 'Pemindai lambat'
    k[per_ip['sub'].isin(sub_scan)] = 'Pemindaian terdistribusi'
    k[k.index.isin(intens)] = 'Serangan intens'
    return g['Source IP'].map(k).fillna('Server (keluar)')

batas = pd.Timestamp(TANGGAL_UJI, tz='UTC')
latih_df, uji_df = df[df.Waktu < batas].copy(), df[df.Waktu >= batas].copy()
latih_df['Kategori'], uji_df['Kategori'] = kategori(latih_df), kategori(uji_df)
pd.DataFrame({'latih: aliran': latih_df.Kategori.value_counts(), 'latih: IP': latih_df.groupby('Kategori')['Source IP'].nunique(),
              'uji: aliran': uji_df.Kategori.value_counts(), 'uji: IP': uji_df.groupby('Kategori')['Source IP'].nunique()}).fillna(0).astype(int)

In [ ]:
print('Sumber serangan intens pada periode latih (dikeluarkan dari data latih "dibersihkan"):')
(latih_df[latih_df.Kategori == 'Serangan intens'].groupby('Source IP')
   .agg(aliran=('Flow ID', 'size'), port_berbeda=('Destination Port', 'nunique'),
        ke_ssh=('Destination Port', lambda s: int((s == 22).sum())), mulai=('Waktu', 'min'), selesai=('Waktu', 'max'))
   .sort_values('aliran', ascending=False))

## 4. Training: apa adanya vs dibersihkan

Model dilatih dari **seluruh** lalu lintas normal (koneksi masuk dan keluar), tetapi yang **dinilai** hanya koneksi
yang dimulai dari luar (lalu lintas masuk), sama seperti sistem NIDS di server. Percobaan melatih hanya dengan koneksi
masuk justru menaikkan alarm palsu pada pengunjung (VPS uji 8,5% → 18,3%; VPS perusahaan 3,4% → 3,9%), karena koneksi
keluar ikut memperkaya gambaran "normal".

- **Apa adanya**: semua aliran periode latih.
- **Dibersihkan**: aliran dari sumber *serangan intens* dan subnet *pemindaian terdistribusi* dikeluarkan. Isolation Forest menganggap normal apa pun yang
  sering muncul di data latih; bila pemindaian ikut dilatih, model "terbiasa" dan tidak lagi menandainya
  (efek *masking*, Liu et al., 2008).

In [ ]:
t0 = time.time()
paket_apa = latih(latih_df, C_A, C_C, JENDELA)
DIBUANG = ['Serangan intens', 'Pemindaian terdistribusi']
paket_bersih = latih(latih_df[~latih_df.Kategori.isin(DIBUANG)], C_A, C_C, JENDELA)
for nama, p in (('Apa adanya', paket_apa), ('Dibersihkan', paket_bersih)):
    print('%-12s ambang A = %.4f | ambang C = %.4f' % (nama, p['A']['threshold'], p['C']['threshold']))
print('%.1f detik' % (time.time() - t0))

## 5. Pemeriksaan pada periode uji

Persentase aliran yang ditandai anomali per kategori sumber. Periode latih 22–26 Okt 2026, periode uji 27 Okt 2026 dan seterusnya (UTC).

**Cara membaca:**
- *Serangan intens*, *Pemindaian terdistribusi*, dan *Pemindai lambat*: semakin tinggi semakin baik (serangan tertangkap).
- *Pengunjung berdata*: semakin rendah semakin baik. Ini perkiraan **alarm palsu**.
- *Lainnya*: bot yang hanya mengetuk 1–2 port. Wajar bila rendah, karena satu ketukan bukan pemindaian terhadap server ini.

In [ ]:
def tandai(paket, x):
    m = ModelDeteksi(paket)
    return (m.A.skor(x) > m.A.ambang) | (m.C.skor(x) > m.C.ambang)

uji_df = uji_df[uji_df.Masuk].copy()                            # hanya koneksi dari luar yang dinilai
hasil = {}
for nama, p in (('Apa adanya', paket_apa), ('Dibersihkan', paket_bersih)):
    t = tandai(p, uji_df)
    uji_df['Ditandai_' + nama] = t
    hasil[nama + ' (% aliran)'] = (uji_df.groupby('Kategori')['Ditandai_' + nama].mean() * 100).round(1)
    hasil[nama + ' (IP ditandai)'] = uji_df[t].groupby('Kategori')['Source IP'].nunique()
tabel = pd.DataFrame(hasil).fillna(0)
tabel['IP pada periode uji'] = uji_df.groupby('Kategori')['Source IP'].nunique()
tabel.to_csv(os.path.join(HASIL, 'tabel_pemeriksaan_%s.csv' % NAMA))
print('Total aliran uji: %s | ditandai (dibersihkan): %.2f%%' % (f'{len(uji_df):,}', uji_df['Ditandai_Dibersihkan'].mean() * 100))
tabel

In [ ]:
print('IP yang ditandai (model dibersihkan), untuk diperiksa satu per satu:')
tanda = uji_df[uji_df['Ditandai_Dibersihkan']]
(tanda.groupby(['Kategori', 'Source IP'])
      .agg(ditandai=('Flow ID', 'size'), port_berbeda=('Destination Port', 'nunique'),
           port_teratas=('Destination Port', lambda s: ', '.join(map(str, s.value_counts().index[:4]))))
      .join(uji_df.groupby('Source IP').size().rename('total_aliran'), on='Source IP')
      .sort_values(['Kategori', 'ditandai'], ascending=[True, False]).head(25))

## 6. Model final untuk sistem NIDS

Setelah cara latih dipilih (**dibersihkan**), model final dilatih dari **seluruh** rekaman (latih + uji) yang sudah
dibersihkan, supaya model mengenal pola normal sebanyak mungkin. Hasil pemeriksaan di atas ikut disimpan di dalam
model dan tampil di halaman *Model & evaluasi* pada dashboard.

In [ ]:
df['Kategori'] = kategori(df)
intens_semua = sorted(sumber_intens(df))
bersih = df[~df.Kategori.isin(DIBUANG)]
paket = latih(bersih, C_A, C_C, JENDELA, info={
    'server_ip': sorted(SERVER_IP), 'cara_latih': 'dibersihkan',
    'periode_latih': [str(df.Waktu.min()), str(df.Waktu.max())], 'jumlah_aliran_latih': int(len(bersih)),
    'aturan_pembersihan': {'ambang_scan': LM.AMBANG_SCAN, 'ambang_ssh': LM.AMBANG_SSH}, 'sumber_intens': intens_semua})

rancangan = json.load(open(os.path.join(KODE, 'deploy', 'evaluasi_rancangan.json'), encoding='utf-8'))
t = tabel['Dibersihkan (% aliran)']
paket['evaluasi'] = {'bagian': rancangan['bagian'] + [{
    'judul': 'Pemeriksaan periode uji: %s' % NAMA_SERVER,
    'keterangan': 'Model dilatih pada data sebelum %s (UTC) yang dibersihkan, diperiksa pada data sesudahnya. Label berbasis aturan.' % TANGGAL_UJI,
    'metrik': {'Aliran uji ditandai': '%.2f%%' % (uji_df['Ditandai_Dibersihkan'].mean() * 100),
               **{'%s ditandai' % k: '%.1f%%' % v for k, v in t.items()}}}]}

MODEL = os.path.join(HASIL, 'model_%s.joblib' % NAMA)
joblib.dump(paket, MODEL)
print('Data latih final : %s aliran (%d sumber intens dikeluarkan)' % (f'{len(bersih):,}', len(intens_semua)))
print('Ambang A = %.4f | ambang C = %.4f | versi %s' % (paket['A']['threshold'], paket['C']['threshold'], paket['versi']))
print('Model disimpan   :', MODEL)

## 7. Memasang model ke server

Model sebaiknya **dilatih ulang di server** dari berkas fitur di atas, supaya versi scikit-learn sama dengan yang
dipakai modul deteksi. Prosedurnya identik dengan bagian 6 (aturan pembersihan yang sama):

```bash
sudo bash deploy/pasang-deteksi.sh --nama-server "NAMA SERVER" --latih fitur_<nama>.parquet
```

Setelah itu modul deteksi berjalan real-time dengan model tersebut, mencatat ke log SQLite, mengirim notifikasi
Telegram, dan hasilnya tampil di dashboard (lihat notebook `05_Cara_Kerja_NIDS.ipynb`).